# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr and **monthly NEE** from the
FLUXNET zarr for stations inside a Netherlands bounding box, restricted
to 2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

In [1]:
import xarray as xr
import zarr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

def in_nl(lat, lon):
    return (lat is not None and lon is not None
            and LAT_MIN <= float(lat) <= LAT_MAX
            and LON_MIN <= float(lon) <= LON_MAX)

## Obspack — CO2

In [2]:
z = zarr.open_group("icos-obspack.zarr", mode="r")

co2 = {}
for sid in sorted(z.group_keys()):
    a = z[sid].attrs
    if not in_nl(a.get("site_latitude"), a.get("site_longitude")):
        continue
    if "co2" not in z[sid]:
        continue
    ds = xr.open_zarr("icos-obspack.zarr", group=sid, consolidated=False)
    co2[sid] = ds["co2"].sel(time_co2=slice(T0, T1))

for sid, da in co2.items():
    print(f"{sid:8s}  n={da.size:>5}  units={da.attrs.get('units','')}  "
          f"mean={float(da.mean()):.2f}")

CBW127    n= 8549  units=ppm  mean=432.71
CBW207    n= 8739  units=ppm  mean=430.82
CBW27     n= 8526  units=ppm  mean=439.16
CBW67     n= 8541  units=ppm  mean=435.18
JUE120    n= 8443  units=ppm  mean=434.02
JUE50     n= 7996  units=ppm  mean=436.78
JUE80     n= 7992  units=ppm  mean=435.31
LUT0      n= 8332  units=ppm  mean=431.81


## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [ ]:
zf = zarr.open_group("icos-fluxnet.zarr", mode="r")

nee = {}
for sid in sorted(zf.group_keys()):
    a = zf[sid].attrs
    if not in_nl(a.get("geospatial_lat"), a.get("geospatial_lon")):
        continue
    ds = xr.open_zarr("icos-fluxnet.zarr", group=f"{sid}/fluxnet_mm", consolidated=False)
    nee[sid] = (ds["NEE"]
                .sel(ustar_threshold="VUT", nee_variant="REF")
                .sel(time=slice(T0, T1)))

for sid, da in nee.items():
    print(f"{sid:8s}  n={da.size:>3}  units={da.attrs.get('units','')}  "
          f"mean={float(da.mean()):.3f}")

## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

Launch the proxy first:

```bash
python run_proxy.py --store-dir . --port 8080
```

In [ ]:
from datapassport_zarr import open_zarr

OBSPACK_URL = "http://localhost:8080/icos-obspack.zarr"
FLUXNET_URL = "http://localhost:8080/icos-fluxnet.zarr"

# Station lists from the local-store query above (need lat/lon to filter, which
# requires reading group .zattrs — easiest is to use the same zarr-on-disk index
# we already built; the proxy serves the same layout).
nl_obspack = sorted(co2)            # e.g. CBW27, CBW67, …, JUE50, …, LUT0
nl_fluxnet = sorted(nee)            # e.g. BE-Bra, BE-Maa, DE-RuS, NL-Loo

print(f"=== Obspack / CO2 via proxy ({len(nl_obspack)} stations) ===")
for sid in nl_obspack:
    with open_zarr(OBSPACK_URL, group=sid, verbose=False) as ds:
        da = ds["co2"].sel(time_co2=slice(T0, T1))
        m  = float(da.mean())
    print(f"  {sid:8s}  n={da.size:>5}  mean={m:.2f} ppm")

print()
print(f"=== Fluxnet / monthly NEE via proxy ({len(nl_fluxnet)} stations) ===")
for sid in nl_fluxnet:
    with open_zarr(FLUXNET_URL, group=f"{sid}/fluxnet_mm", verbose=False) as ds:
        da = (ds["NEE"]
                .sel(ustar_threshold="VUT", nee_variant="REF")
                .sel(time=slice(T0, T1)))
        m  = float(da.mean())
    print(f"  {sid:8s}  n={da.size:>3}  mean={m:.3f} {da['NEE'].attrs.get('units','') if 'NEE' in da.coords else ''}")